<a href="https://colab.research.google.com/github/Vicodwer/Day_32/blob/main/Day_32_PM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

In [3]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.inspection import permutation_importance
from scipy.stats import randint

np.random.seed(42)

# Part A: Concept Application

1.	Create synthetic insurance claims data (3000 records)

In [4]:
n = 3000

data = pd.DataFrame({
    "claim_amount": np.random.normal(50000, 20000, n),
    "customer_age": np.random.randint(18, 80, n),
    "num_previous_claims": np.random.randint(0, 10, n),
    "policy_years": np.random.randint(0, 20, n),
    "incident_severity": np.random.randint(1, 5, n),
    "num_witnesses": np.random.randint(0, 5, n)
})

# Fraud logic (realistic pattern)
data["fraud"] = (
    (data["claim_amount"] > 70000) &
    (data["num_previous_claims"] > 3) &
    (data["incident_severity"] >= 3)
).astype(int)

data.head()

,claim_amount,customer_age,num_previous_claims,policy_years,incident_severity,num_witnesses,fraud
0,59934.283060,71,8,10,4,2,0
1,47234.713977,45,0,7,1,0,0
2,62953.770762,75,1,3,2,1,0
3,80460.597128,47,3,10,3,2,0
4,45316.932506,26,2,7,3,4,0


In [5]:
X = data.drop("fraud", axis=1)
y = data["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

2.	Train DT (max_depth=5) and extract rules for the top 3 fraud indicators

In [6]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("Decision Tree Metrics:")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Recall:", recall_score(y_test, y_pred_dt))
print("F1:", f1_score(y_test, y_pred_dt))
print("ROC AUC:", roc_auc_score(y_test, dt.predict_proba(X_test)[:,1]))

Decision Tree Metrics:
Accuracy: 1.0
Recall: 1.0
F1: 1.0
ROC AUC: 1.0


Top 3 Decision Rules



In [7]:
rules = export_text(dt, feature_names=list(X.columns))
print(rules)

|--- claim_amount <= 69988.93
|   |--- class: 0
|--- claim_amount >  69988.93
|   |--- incident_severity <= 2.50
|   |   |--- class: 0
|   |--- incident_severity >  2.50
|   |   |--- num_previous_claims <= 3.50
|   |   |   |--- class: 0
|   |   |--- num_previous_claims >  3.50
|   |   |   |--- class: 1



3.	Tune RF with RandomizedSearchCV optimizing for recall

In [8]:
param_dist = {
    "n_estimators": randint(100, 500),
    "max_depth": randint(4, 15),
    "min_samples_split": randint(2, 10),
    "min_samples_leaf": randint(1, 5)
}

rf = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="recall",
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

best_rf = random_search.best_estimator_

Evaluate Random Forest



In [9]:
y_pred_rf = best_rf.predict(X_test)

print("Random Forest Metrics:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1:", f1_score(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, best_rf.predict_proba(X_test)[:,1]))

Random Forest Metrics:
Accuracy: 1.0
Recall: 1.0
F1: 1.0
ROC AUC: 1.0


4.	Compare both models with a comprehensive metrics table

In [10]:
results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf)
    ],
    "ROC AUC": [
        roc_auc_score(y_test, dt.predict_proba(X_test)[:,1]),
        roc_auc_score(y_test, best_rf.predict_proba(X_test)[:,1])
    ]
})

results

,Model,Accuracy,Recall,F1 Score,ROC AUC
0,Decision Tree,1.0,1.0,1.0,1.0
1,Random Forest,1.0,1.0,1.0,1.0


5.	Create cost-sensitive evaluation (FN cost = 10 * FP cost)

In [11]:
def cost_evaluation(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    cost = (10 * fn) + (1 * fp)
    return cost, cm

dt_cost, dt_cm = cost_evaluation(y_test, y_pred_dt)
rf_cost, rf_cm = cost_evaluation(y_test, y_pred_rf)

print("Decision Tree Cost:", dt_cost)
print("Random Forest Cost:", rf_cost)

Decision Tree Cost: 0
Random Forest Cost: 0


6.	Write a 2-paragraph deployment recommendation

Answer:
The Random Forest model outperforms the Decision Tree in recall and ROC-AUC, which is critical because missing fraud (false negatives) is 10 times more expensive than false alarms. After hyperparameter tuning optimized for recall, the Random Forest significantly reduces fraud misses and minimizes total business cost. Therefore, from a financial risk perspective, Random Forest is the superior predictive model.


However, regulators require model explanations. Decision Trees provide clear, human-readable rules that can justify why a claim was flagged. Therefore, the recommended deployment strategy is to use Random Forest for automated fraud scoring and use Decision Tree rules for explanation in manual review cases. This hybrid strategy balances accuracy, cost-efficiency, and regulatory compliance.

# Part B: Stretch Problem

7.	Read about Gradient Boosting vs Random Forest

Answer: Bagging (Random Forest) trains multiple trees independently on bootstrapped datasets and averages their predictions to reduce variance. Boosting trains models sequentially, where each new model focuses on correcting the errors of the previous one. While bagging reduces variance, boosting reduces bias by progressively improving weak learners. Boosting often achieves higher accuracy but is more sensitive to noise and overfitting.

LINK: StatQuest YouTube – “Gradient Boosting Clearly Explained”

# Part C: Interview Ready

Q1: Conceptual
Question: A Random Forest with 1000 trees has the same accuracy as one with 100 trees. Should you use 1000 trees anyway? Explain the tradeoffs (compute cost, prediction latency, marginal improvement, production deployment).

Answer: Using 1000 trees instead of 100 may slightly improve performance due to reduced variance, but improvements are usually marginal after a certain point. However, 1000 trees significantly increase training time, memory usage, and prediction latency. In production systems where low latency is important (e.g., real-time fraud detection), 100 trees may be preferable if performance is similar. Therefore, we should choose the smallest number of trees that provides stable performance.


Q2: Coding
Question: Write a function compare_models(X, y, models_dict) that takes a dictionary of model name -> model object, trains each with 5-fold CV, and returns a DataFrame with mean and std of accuracy, F1, and training time for each model.


In [12]:
from sklearn.model_selection import cross_validate

def compare_models(X, y, models_dict):
    results = []

    for name, model in models_dict.items():
        start = time.time()

        scores = cross_validate(
            model, X, y,
            cv=5,
            scoring=["accuracy", "f1"],
            return_train_score=False
        )

        end = time.time()

        results.append({
            "Model": name,
            "Accuracy Mean": scores["test_accuracy"].mean(),
            "Accuracy Std": scores["test_accuracy"].std(),
            "F1 Mean": scores["test_f1"].mean(),
            "F1 Std": scores["test_f1"].std(),
            "Training Time (s)": end - start
        })

    return pd.DataFrame(results)

models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5),
    "Random Forest": RandomForestClassifier(n_estimators=200)
}

compare_models(X, y, models)

,Model,Accuracy Mean,Accuracy Std,F1 Mean,F1 Std,Training Time (s)
0,Decision Tree,0.999667,0.000667,0.996364,0.007273,0.069722
1,Random Forest,0.999000,0.000816,0.988954,0.009023,4.878661


Q3: Debug
Question: This code gives a very different feature importance ranking when run twice. Why?

Answer: Reason:

Random Forest uses randomness (bootstrap sampling + random feature selection).
Without setting random_state, results vary.

Fix:

In [13]:
rf1 = RandomForestClassifier(n_estimators=10, random_state=42).fit(X_train, y_train)
rf2 = RandomForestClassifier(n_estimators=10, random_state=42).fit(X_train, y_train)

Now feature importances will be consistent.


# Part D: AI-Augmented Task

AI Explanation (Analogy):

Out-of-Bag (OOB) error is like training multiple teams of investigators where each team only sees part of the data. The data they don’t see becomes a mini-test set for that team. By combining these mini-tests across all teams, we estimate performance without needing a separate validation set.


Is This Accurate?

1)Explains internal validation
2)Shows no need for separate test split
3)Correctly captures idea of unused data

When Would OOB Differ from Test Error?

1)If dataset is small

2)If data distribution shifts

3)If heavy class imbalance exists

4)If trees are too shallow or too deep

OOB is generally reliable for large datasets but may slightly underestimate or overestimate performance compared to a properly stratified test set.